testing HDF5 conversion to other formats

In [ ]:
# ensure the kernel is working
from sys import version
print(version)

In [ ]:
# import pandas as pd
import h5py
import os
from fod_config import *
from dotenv import load_dotenv
from fod3 import read_narr_lat_lon,validate_latlon,read_one_year,read_narr_timeseries, path_to_narrfile

In [ ]:
load_dotenv()

In [ ]:
os.getenv('NARR_INPUT_DIR')

In [ ]:
narr_input_dir = h5folder = os.getenv('NARR_INPUT_DIR')
 
h5filename = path_to_narrfile(2001, narr_input_dir)
os.path.exists(h5filename)



In [ ]:
with h5py.File(h5filename,'r') as hf:    
    print('List of arrays in this file: \n', list(hf.keys()))

**test read of hdf5 files**

In [ ]:
yr = 2008
idy = 1
idx = 1
pc_1year, ws_1year, wd_1year = read_one_year(yr=yr, idy=idy, idx=idx, narr_input_dir=h5folder)
ws_1year

In [ ]:
def read_one_year_grid(yr:str,narr_input_dir:str):
    """read one year hf5 file, extra 3 datasets
    Files must be named like narr_PSD_1980_BC.h5
    
    Args:
        yr (int): year of data to read, embedded in filename
        idy (int): index of grid y coordinate (North/South)
        idx (int): index of grid x coordinate (East/West)
        narr_input_dir (str): path to NARR input files
    Returns:
        tuple of np arrays: timeseries values for PC, WD and WS from one grid point, all hours
    """
    
    h5f_annual_filename = path_to_narrfile(yr, narr_input_dir)
    h5f = h5py.File(h5f_annual_filename, 'r')
    # extract all values for one year
    # previously filtered at read time, like
    #  pc_1year = h5f['pc'][idy,idx,ts:te]
    pc_1year = h5f['PC']
    ws_1year = h5f['WS']
    wd_1year = h5f['WD']
    return pc_1year, ws_1year, wd_1year

In [ ]:
# test grid read
pc_1year, ws_1year, wd_1year = read_one_year_grid(yr=yr, narr_input_dir=narr_input_dir)
pc_1year.shape


In [ ]:
# size of point data for a year
pc_1year.shape[0] * pc_1year.shape[1]

size of ALL data

In [ ]:
start_year = 1979; end_year = 2009
years = range(start_year, end_year+1)  # range 'stop' is the value after the top value becuse PYTHON
count_all_vals = (years[-1] - years[0]) * (pc_1year.shape[0] * pc_1year.shape[1]) * pc_1year.shape[2]

print( "number of vals in 30 yr  stack" , count_all_vals )
print( "gb" , (count_all_vals * 4)/ (1024**3) )

if we have the memory it is less IO to read everythign into memory but need 

In [ ]:
print( "number of vals in 30 yr point stack" ,  (years[-1] - years[0]) * (pc_1year.shape[0] * pc_1year.shape[1]))

Psuedo code to export this to JSON. 

Note that the fod main model currently assumes there are 2920 values per year and just smushes them all together in a long time series with no time associated with them
in the JSON file should have years for each, and a process for doing the smushing

1. read in all years in 3 giant 3d matrices.   or naively 3 dictionaries with year

PC = {1979: 3d np.array, 1980: 3dnp.array. etc}
WD = etc

matrix of x,ys   or list of x,y combinations from one of the years

2. grid_points 
shape is (277, 349, 2920)
so grid points are x: (0:276) y = rang(0:348) pc_1year.shape[0] * pc_1year.shape[1] -> 96673 points

if we save by grid point that is 96K files.  

each file will be 

3. loop for each grid point x,y
point_vals = {}
for each year
    point_vals[year] = get values for x,y

4. store in AWS: 
best practices https://reintech.io/blog/organizing-data-amazon-s3-best-practices

single bucket, but unique indices depending on how data is read

object name has elements Variable (PC, WD, WS) x, y
based on above better to use x,y first and var name last

`{x}/{y}/PC.json`   <- all years in this file for this coordinate

- program needs to find only 3 object, PC, WD, WS
- each object has 2920 timepoint * 30 yrs = 

single bucket means single point of permisions setting since all data will be
read by same program

```python
# setup aws clie config and IAM user creds in folder
import boto3
# global is easier for this one-off
client = boto3.client('s3')
bucket_name  = "narr_wind_timeseries"


def saveS3(winddata, varname,x,y):
    object_name = f"{x}/{y}/{varname}_{x}_{y}.json"
    d = json.dumps(winddata)
    try:
        res = client.put_object(Body=d, Bucket=bucket_name, Key=object_name)
    except exception as e:
        print(f"can't write {object_name} to bucket")
        raise
    print(res)
```

In [ ]:
# reload .env file to accommodate changes during development
load_dotenv(override = True)
print(os.getenv("REGION_NAME"))
print(os.getenv("NARR_INPUT_DIR"))
os.path.exists(os.getenv("NARR_INPUT_DIR"))

create an AWS session, but ensure to use the AWS config that's in the .env fle or set by the program instead of the default config that may be set in your `~/.aws/config` file

In [ ]:
aws_config = { 
    "aws_access_key_id" : os.getenv("AWS_ACCESS_KEY_ID"), 
    "aws_secret_access_key" : os.getenv("AWS_SECRET_ACCESS_KEY"), 
    "region_name" : os.getenv("REGION_NAME") 
}

import boto3
session = boto3.Session(**aws_config)
session

Make an s3 client from session

In [ ]:
s3_client = session.client('s3')
s3_client

In [ ]:
response = s3_client.list_buckets()
print([bucket['Name'] for bucket in response['Buckets']])

bucket_list = [ bucket['Name'] for bucket in list(response['Buckets'])]
narr_bucket = os.getenv('BUCKET_NAME')
print(narr_bucket in bucket_list)

just for sharing purposes, put all the NARR files into that bucket as they are

In [ ]:
narr_input_dir = os.getenv('NARR_INPUT_DIR')

yr = 1980
narr_file = path_to_narrfile(yr,narr_input_dir)
print(narr_file)
print(os.path.exists(narr_file))


In [ ]:
# copy up all the HDF5 files in current form to S3 
# this is done
# from pathlib import Path

# for file in Path(narr_input_dir).rglob('*.h5'):
#     print(f"uploading {file.name}")
#     s3_client.upload_file(file, narr_bucket, f"narr/{file.name}")

In [ ]:
# read in all years, full grid into dictionaries keyed on year
PC = {}
WD = {}
WS = {}

for yr in range(1979,2009):
    print(yr)
    PC[yr], WD[yr], WS[yr] = read_one_year_grid(yr, narr_input_dir)


In [ ]:
# get config variables - I really need to put this into a module and config class
time_flag = os.getenv("TIME_FLAG")
output_offset_dir=os.getenv("OUTPUT_OFFSET_DIR")
narr_file=os.getenv("NARR_INPUT")
narr_input_dir=os.getenv("NARR_INPUT_DIR")

In [ ]:
LON, LAT = read_narr_lat_lon(narr_file)

In [ ]:
# create a list of all grid coordinate pairs in X,Y space (LON/LAT)
import numpy as np
xdx = list(range(PC[1979].shape[0]))
ydx = list(range(PC[1979].shape[1]))
coords = []

for a in xdx:
    for b in  ydx:
        coords.append( [ a, b ] )
coords

In [ ]:
import json

save_path = "/mnt/research/ICER-RSE/clients/enviroweather/mioffset/maaa_mioffset/tmp"
output_offset_dir=os.getenv("OUTPUT_OFFSET_DIR")
years = range(1979, 2009)

# save stacked point files... 
for coord in coords:     
    print(coord)
    x,y = coord
    pc_years = {}
    wd_years = {}
    ws_years = {}
    for yr in years:
        print(f"\t {yr}")
        wd_years[yr] = list(map(float, WD[yr][x][y]))        
        ws_years[yr] = list(map(float, WS[yr][x][y]))
        pc_years[yr] = list(map(float, PC[yr][x][y]))
        
    
    print(s3_client.put_object(
        Body=json.dumps(pc_years),
        Bucket=narr_bucket,
        Key=f"pc/pc_{x}_{y}.json"
        ))

    print(
        s3_client.put_object(
            Body=json.dumps(wd_years),
            Bucket=narr_bucket,
            Key=f"wd/wd_{x}_{y}.json"
            )
    )

    print(
        s3_client.put_object(
            Body=json.dumps(ws_years),
            Bucket=narr_bucket,
            Key=f"ws/ws_{x}_{y}.json"
        )
    )


    # with open(time_series_file, 'w') as f:
    #     f.writelines(ts_j)


In [ ]:
# test writing and converting to np array
# ts = timeseries
import json
ts_by_year_file = "../tmp/wd_0_10.json"
with open(ts_by_year_file, "r") as f:
    ts_by_year = f.read()

ts_by_year = json.loads(ts_by_year)
ts_by_year


In [ ]:
ts_by_year_nparray = list(map(np.array, list(ts_by_year.values())))
ts_by_year_merged = np.array(np.concatenate(ts_by_year_nparray))
ts_by_year_merged

In [ ]:
s3_client